# Anime Subtitle Generator (WhisperJAV, T4)

One job: download the episode from your server, generate Japanese subtitles with the
anime-tuned pipeline, done.

**Pipeline:** `qwen` + `anime-whisper` (the model WhisperJAV's own docs call
"best-in-class on anime-style dialogue") with WhisperSeg VAD, sensitivity `aggressive`.

**Run order:** Step 1 -> Step 2 -> Step 3. Change the URL or filename only in Step 1.

In [ ]:
#@title Step 1: Settings { display-mode: "form" }

#@markdown ### Source
SOURCE_URL  = "http://violetstreamio.com/MRVIOLETSTREAMBOT-AgAD7R15412" #@param {type:"string"}
TARGET_NAME = "Welsh.and.Shedar.S01E01.1080p.AMZN.WEB-DL.JPN.DDP2.0.H.264-ToonsHub.mkv" #@param {type:"string"}
EXPECTED_MB = 742 #@param {type:"integer"}

#@markdown ### Subtitle output
subtitle_language = "Japanese" #@param ["Japanese", "English (AI translate)"]

#@markdown ### Anime pipeline (defaults are the anime-tuned values)
sensitivity       = "aggressive" #@param ["conservative", "balanced", "aggressive"]
qwen_generator    = "anime-whisper" #@param ["anime-whisper", "qwen3"]
speech_segmenter  = "whisperseg" #@param ["whisperseg", "ten", "firered-vad", "silero-v3.1", "silero-v4.0", "silero-v6.2"]

#@markdown ### Translation (only used when output is "English (AI translate)")
translation_provider = "local" #@param ["local", "gemini", "deepseek", "openrouter", "claude", "gpt", "groq"]
translation_model    = "gemma-9b" #@param {type:"string"}
translation_api_key  = "" #@param {type:"string"}

#@markdown ### Credit line
closing_credit = "Subs by WhisperJAV" #@param {type:"string"}

WHISPERJAV_RELEASE = "v1.9.3"
language_code = "native" if subtitle_language == "Japanese" else "llm"

print("Ready.")
print(f"  Source:      {SOURCE_URL}")
print(f"  Filename:    {TARGET_NAME}")
print(f"  Pipeline:    qwen / {qwen_generator} / {speech_segmenter} / {sensitivity}")
print(f"  Output:      {subtitle_language}")
if subtitle_language != "Japanese":
    print(f"  Translation: {translation_provider} / {translation_model}")

In [ ]:
#@title Step 2: Download the episode from your server { display-mode: "form" }
#@markdown Downloads straight into Colab's fast local disk, not Drive.
#@markdown Uses aria2c with 16 parallel connections if the server supports range requests.

import os, subprocess, shlex, time
from pathlib import Path
from IPython.display import display, HTML

def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")

WORK_DIR = Path("/content/media")
WORK_DIR.mkdir(parents=True, exist_ok=True)
dest_file = WORK_DIR / TARGET_NAME

if dest_file.exists() and dest_file.stat().st_size > (EXPECTED_MB - 50) * 1024 * 1024:
    status(f"Already downloaded: {dest_file.name} ({dest_file.stat().st_size/1e6:.0f} MB)")
    raise SystemExit()

print("Probing server...")
probe = subprocess.run(["curl", "-sI", "--max-time", "20", SOURCE_URL],
                       capture_output=True, text=True)
headers = probe.stdout.lower()
supports_ranges = "accept-ranges: bytes" in headers
content_len = None
for line in headers.splitlines():
    if line.startswith("content-length:"):
        try:
            content_len = int(line.split(":", 1)[1].strip())
        except ValueError:
            pass

if content_len:
    print(f"  Server reports: {content_len/1e6:.1f} MB")
print(f"  Range requests supported: {supports_ranges}")

if supports_ranges:
    if subprocess.run(["which", "aria2c"], capture_output=True).returncode != 0:
        print("Installing aria2...")
        subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
    print("Downloading with aria2c (16 connections)...\n")
    cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M",
           "--file-allocation=none", "--console-log-level=warn",
           "--summary-interval=5", "--auto-file-renaming=false",
           "--allow-overwrite=true", "-d", str(WORK_DIR),
           "-o", TARGET_NAME, SOURCE_URL]
else:
    print("Server does not support range requests — single-stream download.\n")
    cmd = ["wget", "--continue", "--tries=10", "--waitretry=5",
           "--timeout=60", "--progress=dot:giga", "-O", str(dest_file), SOURCE_URL]

print("Command:", shlex.join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode != 0:
    status("Download failed", False)
    raise SystemExit()

if not dest_file.exists():
    alt = WORK_DIR / (TARGET_NAME + ".1")
    if alt.exists():
        alt.rename(dest_file)
    else:
        raise SystemExit(f"File not found: {dest_file}")

size_mb = dest_file.stat().st_size / 1e6
status(f"Downloaded {dest_file.name} ({size_mb:.0f} MB)")
WHISPERJAV_MEDIA_FILE = dest_file
print("\nRun Step 3 next.")

In [ ]:
#@title Step 3: Install WhisperJAV and transcribe { display-mode: "form" }

import os, sys, subprocess, shlex, time
from pathlib import Path
from IPython.display import display, HTML

os.environ['MPLBACKEND'] = 'Agg'
os.environ['TORCH_HUB_TRUST_REPO'] = '1'

def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")

def section(title):
    print(f"\n{'-'*46}\n{title}\n{'-'*46}")

if 'WHISPERJAV_MEDIA_FILE' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 2 first.</b></div>'))
    raise SystemExit()

section("GPU CHECK")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    status("No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.", False)
    raise SystemExit()
status(f"GPU: {gpu.stdout.strip()}")

section(f"INSTALLING WHISPERJAV {WHISPERJAV_RELEASE}")
install_start = time.time()
REPO_URL = "https://github.com/meizhong986/WhisperJAV.git"
REPO_PATH = "/content/WhisperJAV"
SCRIPT_PATH = f"{REPO_PATH}/installer/install_colab.sh"

check = subprocess.run(["python3", "-c", "import whisperjav"], capture_output=True)
if check.returncode == 0:
    status("WhisperJAV already installed (skipping)")
else:
    if not os.path.exists(REPO_PATH):
        print(f"Cloning {WHISPERJAV_RELEASE}...")
        r = subprocess.run(["git", "clone", "--depth", "1", "--branch", WHISPERJAV_RELEASE,
                            REPO_URL, REPO_PATH], capture_output=True, text=True)
        if r.returncode != 0:
            status(f"Clone failed: {r.stderr.strip()}", False)
            raise SystemExit()
    if not os.path.exists(SCRIPT_PATH):
        status(f"Install script not found: {SCRIPT_PATH}", False)
        raise SystemExit()

    env = {**os.environ,
           "PATH": f"{os.environ.get('PATH', '')}:{os.path.expanduser('~/.local/bin')}",
           "WHISPERJAV_REF": WHISPERJAV_RELEASE}
    p = subprocess.Popen(["bash", SCRIPT_PATH], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in iter(p.stdout.readline, ''):
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        status("Installation failed", False)
        raise SystemExit()
status(f"Install finished ({time.time()-install_start:.0f}s)")

section("TRANSCRIBING")
out_dir = WHISPERJAV_MEDIA_FILE.parent
cmd = ["whisperjav", str(WHISPERJAV_MEDIA_FILE),
       "--output-dir", str(out_dir),
       "--mode", "qwen",
       "--qwen-sensitivity", sensitivity,
       "--qwen-generator", qwen_generator,
       "--qwen-segmenter", speech_segmenter,
       "--subs-language", language_code]

print("Command:", shlex.join(cmd), "\n")
p = subprocess.Popen(shlex.join(cmd), shell=True, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
p.wait()

if p.returncode != 0:
    status("Transcription failed", False)
    raise SystemExit()

srts = sorted([f for f in out_dir.glob('*.srt')
               if '.pass1.' not in f.name and '.pass2.' not in f.name])
status(f"Created {len(srts)} subtitle file(s)")
WHISPERJAV_SRTS = srts
WHISPERJAV_OUT_DIR = out_dir

In [ ]:
#@title Step 4: Add credit + save to Drive { display-mode: "form" }
#@markdown Adds the closing credit, then copies the .srt and .mkv to Google Drive/WhisperJAV/.

import shutil, time
from pathlib import Path
from google.colab import drive
from IPython.display import display, HTML

def status(msg, ok=True):
    print(f"{'OK  ' if ok else 'FAIL'} {msg}")

if 'WHISPERJAV_SRTS' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 3 first.</b></div>'))
    raise SystemExit()

for srt in WHISPERJAV_SRTS:
    if closing_credit:
        try:
            content = srt.read_text(encoding='utf-8')
            content += f"\n9999\n23:59:58,000 --> 23:59:59,000\n{closing_credit}\n"
            srt.write_text(content, encoding='utf-8')
        except Exception as e:
            print(f"  credit failed on {srt.name}: {e}")
status(f"Credit added to {len(WHISPERJAV_SRTS)} file(s)")

drive.mount('/content/drive', force_remount=False)
drive_dir = Path("/content/drive/MyDrive/WhisperJAV")
drive_dir.mkdir(parents=True, exist_ok=True)

for f in list(WHISPERJAV_SRTS) + [WHISPERJAV_MEDIA_FILE]:
    target = drive_dir / f.name
    print(f"Copying {f.name} -> Drive...")
    shutil.copy2(f, target)

status(f"Saved {len(WHISPERJAV_SRTS)} .srt + media to Google Drive/WhisperJAV/")
display(HTML(f'<div style="padding:10px;background:#f0fdf4;border-radius:4px;'
             f'border-left:2px solid #16a34a;font-size:12px"><b>Done.</b> '
             f'Files are in your Drive under <code>WhisperJAV/</code>.</div>'))

print("\nDisconnecting in 10 seconds to save GPU credits...")
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass

In [ ]:
#@title Step 5: AI Translation (only if output is "English (AI translate)") { display-mode: "form" }

import os, shlex, subprocess, time
from pathlib import Path
from IPython.display import display, HTML

if 'WHISPERJAV_SRTS' not in dir():
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>Run Step 3 first.</b></div>'))
    raise SystemExit()

if subtitle_language == "Japanese":
    display(HTML('<div style="padding:8px;background:#f0f9ff;border-radius:4px;'
                 'border-left:2px solid #3b82f6;font-size:11px"><b>Skipped:</b> '
                 'output is Japanese only.</div>'))
    raise SystemExit()

if translation_provider != "local" and not translation_api_key:
    display(HTML('<div style="padding:8px;background:#fef2f2;border-radius:4px;color:#991b1b;'
                 'font-size:11px"><b>No API key.</b> Set one in Step 1, or use local.</div>'))
    raise SystemExit()

if translation_provider != "local":
    env_map = {"gemini": "GEMINI_API_KEY", "deepseek": "DEEPSEEK_API_KEY",
               "openrouter": "OPENROUTER_API_KEY", "claude": "ANTHROPIC_API_KEY",
               "gpt": "OPENAI_API_KEY", "groq": "GROQ_API_KEY"}
    os.environ[env_map.get(translation_provider, "API_KEY")] = translation_api_key

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
drive_dir = Path("/content/drive/MyDrive/WhisperJAV")

print(f"Provider: {translation_provider}")
print(f"Files: {len(WHISPERJAV_SRTS)}\n")

for srt in WHISPERJAV_SRTS:
    print(f"Translating {srt.name}...")
    cmd = ["whisperjav-translate", "-i", str(srt),
           "--provider", translation_provider,
           "-t", "english", "--tone", "standard", "--stream"]
    if translation_model:
        cmd += ["--model", translation_model]
    p = subprocess.Popen(shlex.join(cmd), shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(f"  {line}", end='', flush=True)
    p.wait()
    if p.returncode != 0:
        print(f"  FAILED: {srt.name}")

for f in WHISPERJAV_OUT_DIR.glob('*.en.srt'):
    shutil.copy2(f, drive_dir / f.name)

display(HTML('<div style="padding:10px;background:#f0fdf4;border-radius:4px;'
             'border-left:2px solid #16a34a;font-size:12px"><b>Translation done.</b> '
             'English .srt files are in Google Drive/WhisperJAV/.</div>'))

print("\nDisconnecting in 10 seconds...")
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass